# SAM 3 Video Segmentation / Tracking Notebook for Laser Pipeline

This notebook lets you drop in a video, prompt SAM 3 with either a **text concept** like `"person"` / `"guitar"` / `"dancer"` or a **bounding box**, then view the tracked segmentation over time.

It also exports:
- an overlay preview video,
- binary mask frames,
- mask-edge frames,
- contour JSONL with normalized laser-style coordinates in `[-1, 1]`.

Important framing for Babylon: SAM is not really “edge detection.” Use SAM to get a clean object mask, then run contour extraction / Canny / centerline / path simplification on the mask for laser-drawable paths.


In [ ]:
# 1) Install dependencies
# Run this once per environment. Restart the kernel if imports fail after installation.

%pip install -U ultralytics opencv-python matplotlib ipywidgets tqdm numpy

# SAM 3 uses Ultralytics >= 8.3.237.
# SAM 3 weights are NOT auto-downloaded. You need sam3.pt from the official Hugging Face gated weights.
# Put sam3.pt next to this notebook, or set MODEL_PATH below to the full path.


In [ ]:
# 2) Imports

from pathlib import Path
import json
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Video, HTML, clear_output

import ipywidgets as widgets

print("Imports OK")


In [ ]:
# 3) Configuration

MODEL_PATH = "sam3.pt"      # e.g. "/content/sam3.pt" or "./weights/sam3.pt"
OUTPUT_DIR = Path("sam3_laser_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Performance knobs
CONF = 0.25
IMGSZ = 640                 # 640 is a good first test. Increase to 1024 for better masks, slower inference.
HALF = True                 # Use FP16 on CUDA. Set False on CPU/MPS if you hit issues.
MAX_FRAMES = 300            # Safety cap while experimenting. Set None to process the whole video.
PROCESS_EVERY_N_FRAMES = 1  # Set 2/3/4 for faster tests.

# Laser-ish post-processing knobs
MIN_CONTOUR_AREA = 80       # Drop tiny blobs/noise
APPROX_EPSILON_PX = 2.0     # Larger = fewer contour points
EDGE_METHOD = "contour"     # "contour" or "canny"

if not Path(MODEL_PATH).exists():
    print(f"⚠️ MODEL_PATH not found: {MODEL_PATH}")
    print("Download sam3.pt after getting access to the official SAM 3 weights, then set MODEL_PATH.")
else:
    print(f"Found model: {MODEL_PATH}")


In [ ]:
# 4) Upload/select a video and choose prompt mode

upload = widgets.FileUpload(accept=".mp4,.mov,.avi,.mkv,.webm", multiple=False)
video_path_text = widgets.Text(
    value="",
    placeholder="Or paste a local path, e.g. /content/my_video.mp4",
    description="Video path:",
    layout=widgets.Layout(width="700px"),
)

prompt_mode = widgets.ToggleButtons(
    options=[("Text concept prompt", "text"), ("Bounding box prompt", "bbox")],
    value="text",
    description="Prompt:",
)

text_prompt = widgets.Text(
    value="person",
    placeholder='e.g. "dancer", "guitar", "person wearing red shirt"',
    description="Text:",
    layout=widgets.Layout(width="700px"),
)

bbox_x1 = widgets.IntText(value=100, description="x1")
bbox_y1 = widgets.IntText(value=100, description="y1")
bbox_x2 = widgets.IntText(value=300, description="x2")
bbox_y2 = widgets.IntText(value=300, description="y2")
bbox_box = widgets.HBox([bbox_x1, bbox_y1, bbox_x2, bbox_y2])

display(widgets.VBox([
    widgets.HTML("<b>Upload a video or paste a video path.</b>"),
    upload,
    video_path_text,
    prompt_mode,
    text_prompt,
    widgets.HTML("<b>Bounding box coordinates, used only in bbox mode:</b>"),
    bbox_box,
]))

def get_video_path():
    """Save uploaded video to disk or return manually entered path."""
    if video_path_text.value.strip():
        p = Path(video_path_text.value.strip()).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"Video path does not exist: {p}")
        return str(p)

    if len(upload.value) == 0:
        raise ValueError("Upload a video or paste a local video path.")

    # ipywidgets FileUpload differs slightly across versions.
    item = list(upload.value.values())[0] if isinstance(upload.value, dict) else upload.value[0]
    name = item["metadata"]["name"] if "metadata" in item else item["name"]
    content = item["content"]

    out = Path(name)
    out.write_bytes(content)
    return str(out)

print("Widget ready. Choose a video/prompt, then run the next cell.")


In [ ]:
# 5) Preview first frame and adjust bbox if needed

video_path = get_video_path()
cap = cv2.VideoCapture(video_path)
ok, frame_bgr = cap.read()
cap.release()

if not ok:
    raise RuntimeError(f"Could not read first frame from {video_path}")

frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
h, w = frame_rgb.shape[:2]

plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title(f"First frame: {w} x {h}\nUse these pixel coordinates for bbox mode.")
plt.axis("on")
plt.show()

print("Video:", video_path)
print("Frame size:", w, "x", h)
print("Tip: for bbox mode, set x1,y1,x2,y2 around the object in this first frame.")


In [ ]:
# 6) Helper functions: masks -> overlays, edges, and laser-style contours

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p

def result_to_combined_mask(result, frame_shape):
    """Return a uint8 combined mask from an Ultralytics Results object."""
    h, w = frame_shape[:2]
    if result.masks is None:
        return np.zeros((h, w), dtype=np.uint8)

    data = result.masks.data
    if hasattr(data, "detach"):
        data = data.detach().float().cpu().numpy()

    # data shape: [num_masks, mask_h, mask_w]
    if data.ndim == 2:
        data = data[None, ...]

    combined = np.zeros(data.shape[1:], dtype=np.uint8)
    for m in data:
        combined = np.maximum(combined, (m > 0.5).astype(np.uint8))

    if combined.shape[:2] != (h, w):
        combined = cv2.resize(combined, (w, h), interpolation=cv2.INTER_NEAREST)

    return combined * 255

def mask_to_edges(mask_u8, method="contour"):
    """Convert binary mask to crisp drawable edge image."""
    if method == "canny":
        return cv2.Canny(mask_u8, 50, 150)
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    edge = np.zeros_like(mask_u8)
    cv2.drawContours(edge, contours, -1, 255, 1)
    return edge

def contours_to_laser_paths(mask_u8, min_area=80, epsilon_px=2.0):
    """
    Extract simplified contours from a binary mask.

    Coordinates exported:
    - pixel coordinates
    - normalized galvo-style coords:
        x in [-1, 1]
        y in [-1, 1], flipped so positive y is up
    """
    h, w = mask_u8.shape[:2]
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    paths = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area:
            continue

        approx = cv2.approxPolyDP(c, epsilon_px, closed=True)
        pts = approx.reshape(-1, 2)

        if len(pts) < 3:
            continue

        pixel_points = pts.astype(float).tolist()
        galvo_points = []
        for x, y in pts:
            gx = (float(x) / max(w - 1, 1)) * 2.0 - 1.0
            gy = 1.0 - (float(y) / max(h - 1, 1)) * 2.0
            galvo_points.append([gx, gy])

        paths.append({
            "area_px": float(area),
            "num_points": int(len(pts)),
            "pixel_points": pixel_points,
            "galvo_points": galvo_points,
        })

    # largest paths first
    paths.sort(key=lambda p: p["area_px"], reverse=True)
    return paths

def write_mp4_from_frames(frame_paths, out_path, fps):
    first = cv2.imread(str(frame_paths[0]))
    h, w = first.shape[:2]
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for p in frame_paths:
        im = cv2.imread(str(p))
        writer.write(im)
    writer.release()
    return out_path

print("Helpers loaded")


In [ ]:
# 7) Run SAM 3 video segmentation/tracking

# Notes:
# - Text mode uses SAM3VideoSemanticPredictor: track all instances matching text concepts.
# - Bbox mode uses SAM3VideoPredictor: track the specific object(s) you box in frame 1.
# - For SAM 3, you need ultralytics >= 8.3.237 and a local sam3.pt file.

from ultralytics.models.sam import SAM3VideoPredictor, SAM3VideoSemanticPredictor

if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Cannot find {MODEL_PATH}. Download sam3.pt and set MODEL_PATH before running."
    )

video_path = get_video_path()
run_id = Path(video_path).stem + "_" + time.strftime("%Y%m%d_%H%M%S")
run_dir = ensure_dir(OUTPUT_DIR / run_id)
overlay_dir = ensure_dir(run_dir / "overlay_frames")
mask_dir = ensure_dir(run_dir / "mask_frames")
edge_dir = ensure_dir(run_dir / "edge_frames")
contours_jsonl = run_dir / "laser_contours.jsonl"
overlay_video_path = run_dir / "sam3_overlay.mp4"
edges_video_path = run_dir / "mask_edges.mp4"

# Get video FPS
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 24
cap.release()
effective_fps = fps / PROCESS_EVERY_N_FRAMES

overrides = dict(
    conf=CONF,
    task="segment",
    mode="predict",
    imgsz=IMGSZ,
    model=MODEL_PATH,
    half=HALF,
    verbose=False,
)

mode = prompt_mode.value
if mode == "text":
    predictor = SAM3VideoSemanticPredictor(overrides=overrides)
    prompt = [text_prompt.value.strip()]
    if not prompt[0]:
        raise ValueError("Enter a text concept prompt, e.g. 'person' or 'guitar'.")
    results = predictor(source=video_path, text=prompt, stream=True)
    print("Running SAM 3 text concept tracking:", prompt)
else:
    predictor = SAM3VideoPredictor(overrides=overrides)
    bbox = [[float(bbox_x1.value), float(bbox_y1.value), float(bbox_x2.value), float(bbox_y2.value)]]
    results = predictor(source=video_path, bboxes=bbox, stream=True)
    print("Running SAM 3 bbox tracking:", bbox)

overlay_frame_paths = []
edge_frame_paths = []

with contours_jsonl.open("w") as f:
    for i, r in enumerate(tqdm(results, desc="Processing frames")):
        if MAX_FRAMES is not None and i >= MAX_FRAMES:
            break
        if i % PROCESS_EVERY_N_FRAMES != 0:
            continue

        # Original image
        orig = r.orig_img.copy()
        h, w = orig.shape[:2]

        # Ultralytics overlay
        overlay = r.plot()  # BGR ndarray
        overlay_path = overlay_dir / f"{i:06d}.jpg"
        cv2.imwrite(str(overlay_path), overlay)
        overlay_frame_paths.append(overlay_path)

        # Combined binary mask
        mask = result_to_combined_mask(r, orig.shape)
        mask_path = mask_dir / f"{i:06d}.png"
        cv2.imwrite(str(mask_path), mask)

        # Edges from mask
        edges = mask_to_edges(mask, method=EDGE_METHOD)
        edge_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
        edge_path = edge_dir / f"{i:06d}.jpg"
        cv2.imwrite(str(edge_path), edge_bgr)
        edge_frame_paths.append(edge_path)

        # Contour paths for laser pipeline
        paths = contours_to_laser_paths(
            mask,
            min_area=MIN_CONTOUR_AREA,
            epsilon_px=APPROX_EPSILON_PX,
        )

        record = {
            "frame_index": int(i),
            "width": int(w),
            "height": int(h),
            "num_paths": len(paths),
            "paths": paths,
        }
        f.write(json.dumps(record) + "\n")

if not overlay_frame_paths:
    raise RuntimeError("No frames were processed. Try lowering PROCESS_EVERY_N_FRAMES or checking the video.")

write_mp4_from_frames(overlay_frame_paths, overlay_video_path, effective_fps)
write_mp4_from_frames(edge_frame_paths, edges_video_path, effective_fps)

print("Done.")
print("Run directory:", run_dir)
print("Overlay video:", overlay_video_path)
print("Edges video:", edges_video_path)
print("Contours JSONL:", contours_jsonl)


In [ ]:
# 8) View output videos in the notebook

display(HTML("<h3>SAM 3 segmentation/tracking overlay</h3>"))
display(Video(str(overlay_video_path), embed=True, html_attributes="controls muted"))

display(HTML("<h3>Mask edges for laser path extraction</h3>"))
display(Video(str(edges_video_path), embed=True, html_attributes="controls muted"))


In [ ]:
# 9) Inspect one processed frame and its extracted laser paths

sample_idx = 0
sample_overlay = cv2.cvtColor(cv2.imread(str(overlay_frame_paths[sample_idx])), cv2.COLOR_BGR2RGB)
sample_edge = cv2.imread(str(edge_frame_paths[sample_idx]), cv2.IMREAD_GRAYSCALE)

# Read matching JSONL record
records = []
with open(contours_jsonl, "r") as f:
    for line in f:
        records.append(json.loads(line))

sample_record = records[sample_idx]

plt.figure(figsize=(10, 6))
plt.imshow(sample_overlay)
plt.title("Overlay sample")
plt.axis("off")
plt.show()

plt.figure(figsize=(10, 6))
plt.imshow(sample_edge, cmap="gray")
plt.title("Edge sample")
plt.axis("off")
plt.show()

print("Frame:", sample_record["frame_index"])
print("Number of contour paths:", sample_record["num_paths"])
if sample_record["paths"]:
    print("Largest path points:", sample_record["paths"][0]["num_points"])
    print("First 5 galvo coords:", sample_record["paths"][0]["galvo_points"][:5])


## Practical notes for Babylon

For a laser pipeline, the strongest first architecture is usually:

1. **SAM 3 mask tracking** for the object or concept.
2. **Mask cleanup**: morphology open/close, drop tiny blobs, temporal smoothing.
3. **Contour/centerline extraction**:
   - object silhouette: `cv2.findContours`
   - internal details/features: Canny/Sobel on the original frame, masked by SAM output
   - line-art content: skeletonization or centerline tracing can matter more than outer contours
4. **Laser path optimization**:
   - simplify paths with `approxPolyDP`
   - resample to a fixed point budget
   - duplicate points at corners
   - add blanked travel points between paths
   - normalize to galvo coordinates
5. **Export** to ILDA / Pangolin / your internal frame representation.

SAM gives you object separation and tracking. It does not automatically produce beautiful laser-drawable internal feature lines. For that, combine the SAM mask with edge detection or learned line-art extraction.
